In [21]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
print("Environment configured - protobuf set to pure Python mode, TensorFlow disabled for transformers")

Environment configured - protobuf set to pure Python mode, TensorFlow disabled for transformers


In [22]:
# Installation (run once)
# !pip install -q langchain langchain-community langchain-text-splitters
# !pip install -q langchain-google-genai sentence-transformers faiss-cpu
# !pip install -q pypdf

In [23]:
# Notes
# DirectoryLoader + PyPDFLoader reads embedded text-layer from PDFs
# If files are scanned/image-only PDFs, OCR preprocessing is needed before ingestion

In [24]:
# Core imports
import os
import re
import unicodedata
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional

# LangChain
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

print("All libraries imported successfully")

All libraries imported successfully


In [25]:
# Configuration Class
class RAGConfig:
    """Centralized configuration for RAG system"""
    
    # Paths
    BASE_DIR = Path.cwd()
    PDF_DIR = BASE_DIR / "File_PDFs"
    VECTOR_STORE_DIR = BASE_DIR / "vector_store"
    
    # PDF Loading
    PDF_GLOB_PATTERN = "*.pdf"
    
    # Text Processing
    CHUNK_SIZE = 2000
    CHUNK_OVERLAP = 200
    SEPARATORS = ["\n\n", "\n", ". ", " ", ""]
    
    # Embeddings
    EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
    EMBEDDING_DEVICE = "cpu"  # Change to "cuda" if GPU available
    
    # LLM
    LLM_MODEL = "gemini-2.5-flash"
    LLM_TEMPERATURE = 0.2
    LLM_MAX_TOKENS = 1024
    
    # Retrieval
    RETRIEVAL_K = 6  # Increase recall to avoid missing answer-bearing chunks
    
    # API Keys (Set via environment variables)
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "YOUR_API_KEY_HERE")

config = RAGConfig()
print(" Configuration loaded")
print(f" Base directory: {config.BASE_DIR}")
print(f" PDF directory: {config.PDF_DIR}")

 Configuration loaded
 Base directory: a:\RAG-SGU
 PDF directory: a:\RAG-SGU\File_PDFs


## 2️⃣ Data Ingestion - DirectoryLoader (PyPDFLoader)

In [26]:
def clean_text(text: str) -> str:
    """Clean and normalize text."""
    if not text:
        return ""
    normalized = unicodedata.normalize("NFC", text)
    return re.sub(r"\s+", " ", normalized).strip()


def build_directory_loader(directory: Path, pattern: str = "*.pdf") -> DirectoryLoader:
    """Create a DirectoryLoader configured for PDF files."""
    return DirectoryLoader(
        str(directory),
        glob=pattern,
        loader_cls=PyPDFLoader,
        show_progress=True,
        use_multithreading=True,
        loader_kwargs={"extract_images": False},
    )


def load_pdfs_with_directory_loader(
    directory: Path,
    pattern: str = "*.pdf",
    verbose: bool = True,
) -> Dict[str, str]:
    """Load and merge all PDF pages by filename using DirectoryLoader."""
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory}")

    loader = build_directory_loader(directory, pattern=pattern)

    if verbose:
        print(f"\n{'='*70}")
        print(f"📁 Processing Directory: {directory.name}")
        print(f"{'='*70}")

    docs = loader.load()

    if not docs:
        print(f"⚠️ No PDF content found in {directory}")
        return {}

    grouped_pages: Dict[str, List[str]] = defaultdict(list)
    for doc in docs:
        source_name = Path(str(doc.metadata.get("source", "unknown.pdf"))).name
        content = clean_text(doc.page_content)
        if content:
            grouped_pages[source_name].append(content)

    results = {
        filename: clean_text("\n".join(pages))
        for filename, pages in grouped_pages.items()
        if pages
    }

    if verbose:
        successful = len(results)
        total_chars = sum(len(text) for text in results.values())
        print(f"   Files with extracted text: {successful}")
        print(f"   Total characters: {total_chars:,}")
        print(f"{'='*70}\n")

    return results


def merge_texts(texts_dict: Dict[str, str], add_separators: bool = True) -> str:
    """Merge texts from multiple PDFs."""
    if not texts_dict:
        return ""

    combined_text = []
    for filename, text in texts_dict.items():
        if not text:
            continue
        if add_separators:
            combined_text.append(
                f"\n\n{'='*50}\nDOCUMENT: {filename}\n{'='*50}\n\n{text}"
            )
        else:
            combined_text.append(text)

    return clean_text("\n\n".join(combined_text))


print("✅ DirectoryLoader pipeline initialized (PyPDFLoader)")

✅ DirectoryLoader pipeline initialized (PyPDFLoader)


In [27]:
# Process ALL PDF documents in directory with DirectoryLoader
print(f"📂 Target directory: {config.PDF_DIR}")

if config.PDF_DIR.exists():
    texts_dict = load_pdfs_with_directory_loader(
        config.PDF_DIR,
        pattern=config.PDF_GLOB_PATTERN,
        verbose=True,
    )

    # Keep only successfully extracted documents
    normalized_texts_dict: Dict[str, str] = {
        str(filename): str(content)
        for filename, content in texts_dict.items()
        if isinstance(content, str) and content.strip()
    }

    if not normalized_texts_dict:
        print("⚠️ No extractable text found from PDF directory using DirectoryLoader/PyPDFLoader.")
        print("   This usually means PDFs are scanned/image-only and have no embedded text-layer.")
        print("   Fallback: skip rebuild from PDFs and try loading existing vector store in later cells.")
        texts_dict = {}
        raw_text = ""
    else:
        texts_dict = normalized_texts_dict
        raw_text = merge_texts(texts_dict, add_separators=True)

        # Statistics
        print(f"📊 FINAL STATISTICS:")
        print(f"{'='*70}")
        print(f"   Total documents: {len(texts_dict)}")
        print(f"   Total characters: {len(raw_text):,}")
        print(f"   Total words (approx): {len(raw_text.split()):,}")

        # Document breakdown
        print(f"\n   Documents processed:")
        for filename, text in texts_dict.items():
            status = "✅" if text else "❌"
            char_count = len(text) if text else 0
            print(f"      {status} {filename}: {char_count:,} chars")

        print(f"\n{'='*70}")
        print(f"\n📝 Preview of combined text:")
        print(raw_text[:500] + "...\n")

else:
    print(f"❌ PDF directory not found: {config.PDF_DIR}")
    print("   Please create directory and add PDF files")
    raw_text = ""

# Alternative: Process single file (uncomment if needed)
# texts_dict = load_pdfs_with_directory_loader(config.PDF_DIR, pattern="*specific_file*.pdf")
# raw_text = merge_texts(texts_dict, add_separators=True)

📂 Target directory: a:\RAG-SGU\File_PDFs

📁 Processing Directory: File_PDFs


100%|██████████| 3/3 [00:00<00:00, 19.96it/s]

   Files with extracted text: 0
   Total characters: 0

⚠️ No extractable text found from PDF directory using DirectoryLoader/PyPDFLoader.
   This usually means PDFs are scanned/image-only and have no embedded text-layer.
   Fallback: skip rebuild from PDFs and try loading existing vector store in later cells.


### 📁 Hỗ trợ xử lý nhiều PDF files với DirectoryLoader

**Cách sử dụng:**
```python
# Xử lý cả thư mục
texts_dict = load_pdfs_with_directory_loader(config.PDF_DIR)

# Hoặc lọc theo pattern
texts_dict = load_pdfs_with_directory_loader(config.PDF_DIR, pattern="*CNTT*.pdf")
raw_text = merge_texts(texts_dict, add_separators=True)
```

**Lưu ý:**
- DirectoryLoader + PyPDFLoader đọc text-layer có sẵn trong PDF
- PDF scan ảnh có thể trả về rỗng nếu không có OCR

In [28]:
# 🔧 ADVANCED: Custom processing options (Optional)

# Option 1: Process specific files only
# specific_files = ["file1.pdf", "file2.pdf"]
# all_texts = load_pdfs_with_directory_loader(config.PDF_DIR)
# texts_dict = {name: all_texts[name] for name in specific_files if name in all_texts}

# Option 2: Filter by filename pattern
# texts_dict = load_pdfs_with_directory_loader(config.PDF_DIR, pattern="*CNTT*.pdf")

# Option 3: Process without document separators (for single-document feel)
# raw_text = merge_texts(texts_dict, add_separators=False)

print("💡 Tip: Uncomment above code blocks to use custom processing options")

💡 Tip: Uncomment above code blocks to use custom processing options


## 3️⃣ Text Chunking & Processing

In [29]:
class TextChunker:
    """Handle text splitting into chunks"""
    
    def __init__(self, config: RAGConfig):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.CHUNK_SIZE,
            chunk_overlap=config.CHUNK_OVERLAP,
            separators=config.SEPARATORS,
            length_function=len
        )
    
    def split_text(self, text: str, verbose: bool = True) -> List[str]:
        """Split text into chunks"""
        if not text or len(text) < 100:
            raise ValueError("Input text is too short or empty")
        
        chunks = self.splitter.split_text(text)
        
        if verbose:
            print(f"\n Text Chunking:")
            print(f"   Total chunks: {len(chunks)}")
            print(f"   Avg chunk size: {sum(len(c) for c in chunks) // len(chunks)} chars")
            print(f"   Min/Max: {min(len(c) for c in chunks)} / {max(len(c) for c in chunks)} chars")
        
        return chunks

# Create chunks
if raw_text and len(raw_text) >= 100:
    chunker = TextChunker(config)
    chunks = chunker.split_text(raw_text)

    print(f"\n Sample chunk:")
    print(f"{chunks[0][:200]}...")
else:
    chunks = []
    print("⚠️ No raw text available from ingestion.")
    print("   Will skip chunk creation and try loading existing vector store in Cell 16.")

⚠️ No raw text available from ingestion.
   Will skip chunk creation and try loading existing vector store in Cell 16.


## 4️⃣ Embedding Model & Vector Store

In [30]:
class VectorStoreManager:
    """Manage embeddings and vector store"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self.embeddings = None
        self.vector_store = None
        self._initialize_embeddings()
    
    def _initialize_embeddings(self):
        """Initialize embedding model"""
        print(f"\n Loading embedding model: {self.config.EMBEDDING_MODEL}")
        
        # Keep transformers on the PyTorch path to avoid TensorFlow ABI issues on Windows.
        os.environ.setdefault('USE_TF', '0')
        os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
        
        try:
            self.embeddings = HuggingFaceEmbeddings(
                model_name=self.config.EMBEDDING_MODEL,
                model_kwargs={'device': self.config.EMBEDDING_DEVICE},
                encode_kwargs={'normalize_embeddings': True}
            )
        except TypeError as exc:
            if "Unable to convert function return value to a Python type" in str(exc):
                raise RuntimeError(
                    "Embedding model init failed because TensorFlow was loaded. "
                    "Restart kernel and run from the first setup cell."
                ) from exc
            raise
        
        print(" Embedding model loaded")
    
    def create_vector_store(self, texts: List[str], verbose: bool = True) -> FAISS:
        """Create FAISS vector store from texts"""
        if verbose:
            print(f"\n Building vector store from {len(texts)} chunks...")
        
        self.vector_store = FAISS.from_texts(
            texts=texts,
            embedding=self.embeddings
        )
        
        if verbose:
            print(" Vector store created successfully")
        
        return self.vector_store
    
    def save_vector_store(self, path: Optional[str] = None):
        """Persist vector store to disk"""
        if self.vector_store is None:
            raise ValueError("No vector store to save")
        
        save_path = path or str(self.config.VECTOR_STORE_DIR)
        os.makedirs(save_path, exist_ok=True)
        
        self.vector_store.save_local(save_path)
        print(f" Vector store saved to: {save_path}")
    
    def load_vector_store(self, path: Optional[str] = None) -> FAISS:
        """Load vector store from disk"""
        load_path = path or str(self.config.VECTOR_STORE_DIR)
        
        if not os.path.exists(load_path):
            raise FileNotFoundError(f"Vector store not found at: {load_path}")
        
        self.vector_store = FAISS.load_local(
            load_path,
            self.embeddings,
            allow_dangerous_deserialization=True
        )
        print(f" Vector store loaded from: {load_path}")
        return self.vector_store

# Initialize vector store manager
vector_manager = VectorStoreManager(config)


 Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
 Embedding model loaded


In [31]:
# Smoke test for embedding + FAISS lifecycle
sample_chunks = [
    "Ngành Công nghệ thông tin có mục tiêu đào tạo kỹ sư phần mềm.",
    "Chương trình bao gồm học phần lập trình, cơ sở dữ liệu và AI.",
    "Sinh viên có thể làm việc ở vị trí backend, frontend, data engineer.",
]

smoke_store = vector_manager.create_vector_store(sample_chunks, verbose=False)
docs = smoke_store.similarity_search("mục tiêu đào tạo ngành CNTT", k=2)
assert len(docs) == 2, "Similarity search returned unexpected number of docs"

smoke_path = config.VECTOR_STORE_DIR / "smoke_test"
vector_manager.save_vector_store(str(smoke_path))
_ = vector_manager.load_vector_store(str(smoke_path))

import shutil
shutil.rmtree(smoke_path, ignore_errors=True)
print("Embedding/vector store smoke test passed")

 Vector store saved to: a:\RAG-SGU\vector_store\smoke_test
 Vector store loaded from: a:\RAG-SGU\vector_store\smoke_test
Embedding/vector store smoke test passed


In [32]:
# Create and save vector store (or load existing one when no chunks were created)
if chunks:
    vector_store = vector_manager.create_vector_store(chunks)

    # Save for future use
    try:
        vector_manager.save_vector_store()
    except Exception as e:
        print(f" Could not save vector store: {e}")
else:
    print("⚠️ No chunks to index from current PDF ingestion.")
    print(f"   Trying to load existing vector store from: {config.VECTOR_STORE_DIR}")
    try:
        vector_store = vector_manager.load_vector_store()
    except Exception as e:
        raise RuntimeError(
            "No chunks were created and existing vector store could not be loaded."
        ) from e

⚠️ No chunks to index from current PDF ingestion.
   Trying to load existing vector store from: a:\RAG-SGU\vector_store
 Vector store loaded from: a:\RAG-SGU\vector_store


In [33]:
# Test similarity search
test_query = "Mục tiêu đào tạo của ngành Công nghệ thông tin là gì?"
print(f"\n Testing retrieval with query: '{test_query}'\n")

relevant_docs = vector_store.similarity_search(test_query, k=3)

for i, doc in enumerate(relevant_docs, 1):
    print(f"\n{'='*60}")
    print(f" Document {i}:")
    print(f"{'='*60}")
    print(doc.page_content[:300] + "...")


 Testing retrieval with query: 'Mục tiêu đào tạo của ngành Công nghệ thông tin là gì?'


 Document 1:
IELTS = 5.5 do IDP Educa- tion hoac British Council cap...

 Document 2:
IELTS = 5.0 do IDP Education hoac_ British Council cap...

 Document 3:
mang Internet....


## 5️⃣ LLM Configuration - Gemini API

In [34]:
class LLMManager:
    """Manage LLM initialization and configuration"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self.llm = None
    
    def initialize_gemini(self, api_key: Optional[str] = None) -> ChatGoogleGenerativeAI:
        """Initialize Google Gemini LLM"""
        api_key = api_key or self.config.GOOGLE_API_KEY
        
        if not api_key or api_key == "YOUR_API_KEY_HERE":
            raise ValueError(
                "❌ Google API Key not configured!\n"
                "Get your free API key at: https://makersuite.google.com/app/apikey\n"
                "Then set environment variable GOOGLE_API_KEY or pass api_key directly"
            )
        
        print(f"\n⏳ Initializing Gemini ({self.config.LLM_MODEL})...")
        
        self.llm = ChatGoogleGenerativeAI(
            model=self.config.LLM_MODEL,
            google_api_key=api_key,
            temperature=self.config.LLM_TEMPERATURE,
            max_output_tokens=self.config.LLM_MAX_TOKENS,
            convert_system_message_to_human=True
        )
        
        print("✅ Gemini LLM initialized successfully")
        print(f"   Model: {self.config.LLM_MODEL}")
        print(f"   Temperature: {self.config.LLM_TEMPERATURE}")
        print(f"   Max tokens: {self.config.LLM_MAX_TOKENS}")
        
        return self.llm

# Initialize LLM manager
llm_manager = LLMManager(config)

# Initialize Gemini with API key from environment variable
# If GOOGLE_API_KEY is not set in .env, you can pass it directly:
# llm = llm_manager.initialize_gemini("your-api-key-here")

try:
    llm = llm_manager.initialize_gemini()  # Auto-load from config.GOOGLE_API_KEY
except ValueError as e:
    print(str(e))
    print("\n💡 Solutions:")
    print("   1. Create .env file with: GOOGLE_API_KEY=your-key")
    print("   2. Or pass directly: llm = llm_manager.initialize_gemini('your-key')")


⏳ Initializing Gemini (gemini-2.5-flash)...
✅ Gemini LLM initialized successfully
   Model: gemini-2.5-flash
   Temperature: 0.2
   Max tokens: 1024


## 6️⃣ RAG Pipeline - Chain Creation

In [35]:
class RAGPipeline:
    """Complete RAG pipeline with retrieval and generation"""
    
    NOT_FOUND_FALLBACK = "Tôi không tìm thấy thông tin này trong tài liệu"
    
    def __init__(self, llm, vector_store, config: RAGConfig):
        self.llm = llm
        self.vector_store = vector_store
        self.config = config
        self.qa_chain = None
        self._build_prompt_template()
        self._create_qa_chain()
    
    @staticmethod
    def _normalize_text(text: str) -> str:
        lowered = text.casefold()
        without_marks = "".join(
            ch for ch in unicodedata.normalize("NFD", lowered) if unicodedata.category(ch) != "Mn"
        )
        return re.sub(r"[^a-z0-9]+", " ", without_marks).strip()
    
    def _is_not_found_answer(self, answer: str) -> bool:
        return self._normalize_text(answer) == self._normalize_text(self.NOT_FOUND_FALLBACK)
    
    def _build_retriever(self, k: Optional[int] = None):
        effective_k = int(k or self.config.RETRIEVAL_K)
        fetch_k = max(effective_k * 4, effective_k + 8)
        # MMR helps reduce duplicate/noisy chunks from OCR-heavy corpora
        return self.vector_store.as_retriever(
            search_type="mmr",
            search_kwargs={"k": effective_k, "fetch_k": fetch_k, "lambda_mult": 0.7},
        )
    
    def _build_prompt_template(self):
        """Create optimized prompt template"""
        template = """Bạn là trợ lý AI chuyên nghiệp, chuyên gia về tài liệu đào tạo.

NHIỆM VỤ: Trả lời câu hỏi của người dùng dựa trên thông tin từ tài liệu được cung cấp.

THÔNG TIN TỪ TÀI LIỆU:
{context}

CÂU HỎI: {question}

YÊU CẦU:
1. Trả lời chính xác, dựa hoàn toàn vào thông tin được cung cấp
2. Trả lời ngắn gọn, rõ ràng bằng tiếng Việt
3. Nếu không tìm thấy thông tin, hãy trả lời: "Tôi không tìm thấy thông tin này trong tài liệu"
4. Không bịa đặt thông tin không có trong tài liệu
5. Sử dụng bullet points nếu cần liệt kê
6. Nếu câu hỏi không liên quan đến tài liệu, trả lời: "Câu hỏi này không liên quan đến tài liệu đã cho"

TRẢ LỜI:"""
        
        self.prompt = PromptTemplate(
            template=template,
            input_variables=["context", "question"]
        )
    
    def _create_qa_chain(self, k: Optional[int] = None):
        """Create RetrievalQA chain"""
        effective_k = int(k or self.config.RETRIEVAL_K)
        print(f"\n Building RAG chain (k={effective_k}, search=mmr)...")
        
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=self._build_retriever(effective_k),
            chain_type_kwargs={"prompt": self.prompt},
            return_source_documents=True
        )
        
        print(" RAG chain created successfully")
    
    def query(self, question: str, verbose: bool = True) -> Dict:
        """Execute query through RAG pipeline"""
        if not self.qa_chain:
            raise RuntimeError("QA chain not initialized")
        
        if verbose:
            print(f"\n{'='*70}")
            print(f" QUESTION: {question}")
            print(f"{'='*70}")
            print(" Processing...\n")
        
        try:
            result = self.qa_chain.invoke({"query": question})
            answer_text = str(result.get("result", "")).strip()
            source_docs = result.get("source_documents", [])
            
            # Retry once with broader retrieval if model falls back while context exists.
            if self._is_not_found_answer(answer_text) and source_docs:
                retry_k = max(self.config.RETRIEVAL_K + 3, 8)
                if verbose:
                    print(f"⚠️ Fallback detected with {len(source_docs)} source docs -> retry with k={retry_k}")
                retry_chain = RetrievalQA.from_chain_type(
                    llm=self.llm,
                    chain_type="stuff",
                    retriever=self._build_retriever(retry_k),
                    chain_type_kwargs={"prompt": self.prompt},
                    return_source_documents=True
                )
                retry_result = retry_chain.invoke({"query": question})
                retry_answer = str(retry_result.get("result", "")).strip()
                if retry_answer and not self._is_not_found_answer(retry_answer):
                    result = retry_result
                    answer_text = retry_answer
                    source_docs = result.get("source_documents", [])
                    if verbose:
                        print("✅ Retry recovered a grounded answer")
            
            if not answer_text:
                answer_text = self.NOT_FOUND_FALLBACK
                result["result"] = answer_text
            
            if verbose:
                print(f"{'='*70}")
                print(" ANSWER:")
                print(f"{'='*70}")
                print(result['result'])
                
                print(f"\n{'='*70}")
                print(" SOURCE DOCUMENTS:")
                print(f"{'='*70}")
                for i, doc in enumerate(source_docs, 1):
                    print(f"\n[Document {i}]")
                    print(doc.page_content[:200] + "...")
                print(f"\n{'='*70}\n")
            
            return result
            
        except Exception as e:
            print(f" Error during query: {e}")
            raise
    
    def batch_query(self, questions: List[str]) -> List[Dict]:
        """Process multiple questions"""
        results = []
        for i, q in enumerate(questions, 1):
            print(f"\n🔄 Processing question {i}/{len(questions)}")
            result = self.query(q, verbose=False)
            results.append(result)
            print(f"✅ Question {i} completed")
        return results

# Initialize RAG pipeline
if 'llm' in dir() and llm is not None:
    rag_pipeline = RAGPipeline(llm, vector_store, config)
    print("\n🎉 RAG System is ready!")
else:
    print("⚠️ Please initialize LLM first (set API key in previous cell)")


 Building RAG chain (k=6, search=mmr)...
 RAG chain created successfully

🎉 RAG System is ready!


## 7️⃣ Usage Examples & Testing

In [36]:
# Single query example
question = "Mục tiêu đào tạo của ngành Công nghệ thông tin là gì?"
result = rag_pipeline.query(question)


 QUESTION: Mục tiêu đào tạo của ngành Công nghệ thông tin là gì?
 Processing...

⚠️ Fallback detected with 6 source docs -> retry with k=9
 ANSWER:
Tôi không tìm thấy thông tin này trong tài liệu.

 SOURCE DOCUMENTS:

[Document 1]
IELTS = 5.5 do IDP Educa- tion hoac British Council cap...

[Document 2]
mang Internet....

[Document 3]
Cc phuong phdp dénh gid dugc str dung trong CTDT nganh CNTT duge chia thanh 2 nhém chinh: Danh gid tién trinh (On-going/ Formative Assessment) va Danh gid tong két/ dinh ky (Summative Assessment)....

[Document 4]
vpkentt@sgu.edu.vn...

[Document 5]
opt...

[Document 6]
ETS cap....




In [37]:
# Batch queries example
questions = [
    "Giới thiệu về chương trình đào tạo ngành Công nghệ thông tin?",
    "Thời gian đào tạo là bao lâu?",
    "Sinh viên sẽ học những môn học nào?",
    "Cơ hội nghề nghiệp sau khi tốt nghiệp?"
]

print("\n Processing batch queries...\n")
results = rag_pipeline.batch_query(questions)

print("\n" + "="*70)
print(" BATCH RESULTS SUMMARY")
print("="*70)
for i, (q, r) in enumerate(zip(questions, results), 1):
    print(f"\nQ{i}: {q}")
    print(f"A{i}: {r['result'][:150]}...")
    print("-" * 70)


 Processing batch queries...


🔄 Processing question 1/4
✅ Question 1 completed

🔄 Processing question 2/4
✅ Question 2 completed

🔄 Processing question 3/4
✅ Question 3 completed

🔄 Processing question 4/4


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 5.712773678s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_value: 5
}
, retry_d

✅ Question 4 completed

 BATCH RESULTS SUMMARY

Q1: Giới thiệu về chương trình đào tạo ngành Công nghệ thông tin?
A1: Chương trình kỹ sư Công nghệ thông tin:
*   Cụ thể hóa các kiến thức thành các chuẩn đầu ra cấp CTDT (Programme Learning Outcomes – PLOs) dựa trên yêu...
----------------------------------------------------------------------

Q2: Thời gian đào tạo là bao lâu?
A2: Tôi không tìm thấy thông tin này trong tài liệu....
----------------------------------------------------------------------

Q3: Sinh viên sẽ học những môn học nào?
A3: Sinh viên sẽ học về:
*   Word
*   Các thao tác cơ bản trên máy tính
*   Các chủ đề về khoa học máy tính: Thuật toán tiến hóa, Khai phá dữ liệu, Logic ...
----------------------------------------------------------------------

Q4: Cơ hội nghề nghiệp sau khi tốt nghiệp?
A4: Sau khi tốt nghiệp chương trình đào tạo, có cơ hội nghề nghiệp là:

*   Trợ lý bác sĩ...
----------------------------------------------------------------------


In [38]:
# Interactive query function
def ask(question: str):
    """Convenience function for quick queries"""
    return rag_pipeline.query(question)

# Example usage:
# ask("Chương trình học có những học phần nào?")

## 8️⃣ System Utilities & Maintenance

In [39]:
# System information
def print_system_info():
    """Display system configuration and status"""
    print("\n" + "="*70)
    print(" RAG SYSTEM INFORMATION")
    print("="*70)
    print(f"\n Configuration:")
    print(f"   Base Directory: {config.BASE_DIR}")
    print(f"   PDF Directory: {config.PDF_DIR}")
    print(f"   Vector Store: {config.VECTOR_STORE_DIR}")
    print(f"\n Models:")
    print(f"   Embedding: {config.EMBEDDING_MODEL}")
    print(f"   LLM: {config.LLM_MODEL}")
    print(f"\n Parameters:")
    print(f"   Chunk Size: {config.CHUNK_SIZE}")
    print(f"   Chunk Overlap: {config.CHUNK_OVERLAP}")
    print(f"   Retrieval K: {config.RETRIEVAL_K}")
    print(f"\n Data:")
    if 'chunks' in dir():
        print(f"   Total Chunks: {len(chunks)}")
        print(f"   Total Characters: {sum(len(c) for c in chunks):,}")
    print(f"\n Status: {'Active' if 'rag_pipeline' in dir() else 'Not initialized'}")
    print("="*70)

print_system_info()


 RAG SYSTEM INFORMATION

 Configuration:
   Base Directory: a:\RAG-SGU
   PDF Directory: a:\RAG-SGU\File_PDFs
   Vector Store: a:\RAG-SGU\vector_store

 Models:
   Embedding: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
   LLM: gemini-2.5-flash

 Parameters:
   Chunk Size: 2000
   Chunk Overlap: 200
   Retrieval K: 6

 Data:

 Status: Not initialized


In [40]:
# Reload vector store from disk (if previously saved)
def reload_system():
    """Reload vector store without reprocessing PDFs"""
    global vector_store, rag_pipeline
    
    try:
        vector_store = vector_manager.load_vector_store()
        
        if 'llm' in dir() and llm is not None:
            rag_pipeline = RAGPipeline(llm, vector_store, config)
            print(" System reloaded successfully")
        else:
            print(" Vector store loaded, but LLM not initialized")
    except Exception as e:
        print(f"❌ Reload failed: {e}")

# Uncomment to reload:
# reload_system()